# Aula 3, IA simbólica

Notebook executável que acompanha a aula [03-ia-simbolica.md](../../lessons/modulo-01-introducao-ia/03-ia-simbolica.md).

## O que vamos fazer aqui

Vamos construir um pequeno tutor simbólico que recomenda o próximo passo de estudo de
um aluno. O coração dele é um motor de inferência por encadeamento para frente, que
parte de alguns fatos e aplica regras até descobrir tudo o que pode concluir.

Depois, no projeto, damos ao tutor a capacidade de explicar o porquê de cada
recomendação, traçando a cadeia de regras que a sustenta. Essa transparência é uma
das maiores forças da IA simbólica. Tudo aqui é Python puro, sem dependências.

## Base de regras e motor de inferência

Cada regra é um par de condições e uma conclusão. A regra dispara quando todas as
suas condições já estão entre os fatos conhecidos. O motor repete esse processo até
que nenhuma regra nova possa disparar, um estado chamado de ponto fixo.

In [ ]:
# Cada regra é um par (condições, conclusão).
# Todas as condições precisam estar nos fatos para a regra disparar.
regras = [
    (['domina aritmética'], 'pode estudar álgebra'),
    (['domina álgebra'], 'pode estudar funções'),
    (['domina funções'], 'pode estudar cálculo'),
    (['dificuldade em frações'], 'revisar aritmética'),
    (['pode estudar álgebra', 'gosta de desafios'],
     'sugerir problemas avançados de álgebra'),
]


def encadeamento_para_frente(fatos_iniciais, regras):
    """Aplica as regras aos fatos até nenhuma nova conclusão surgir."""
    fatos = set(fatos_iniciais)
    houve_mudanca = True
    while houve_mudanca:
        houve_mudanca = False
        for condicoes, conclusao in regras:
            if conclusao not in fatos and all(c in fatos for c in condicoes):
                fatos.add(conclusao)
                houve_mudanca = True
                print('Regra disparou, novo fato:', conclusao)
    return fatos

## Rodando o tutor

Vamos partir de dois fatos sobre um aluno: ele domina aritmética e gosta de desafios.
O motor descobre o que se segue disso.

In [ ]:
fatos_iniciais = ['domina aritmética', 'gosta de desafios']
resultado = encadeamento_para_frente(fatos_iniciais, regras)

print()
print('Fatos finais:')
for fato in sorted(resultado):
    print(' -', fato)

A partir de dois fatos, o motor deduz que o aluno pode estudar álgebra e que
vale sugerir problemas avançados, conclusão que só aparece porque ele domina
aritmética e gosta de desafios ao mesmo tempo. Repare que a cadeia para aí: como
temos pode estudar álgebra, e não domina álgebra, a regra do próximo nível não
dispara, o que é coerente com a ideia de pré-requisitos.

## Projeto da aula, explicando o porquê

Um sistema simbólico consegue justificar cada decisão. Vamos guardar, para cada
conclusão, qual regra a gerou, e então percorrer essa cadeia de trás para frente para
montar a explicação. Pedimos a justificativa de um fato realmente derivado.

In [ ]:
def encadeamento_com_justificativa(fatos_iniciais, regras):
    fatos = set(fatos_iniciais)
    origem = {f: None for f in fatos_iniciais}  # fato inicial não tem regra de origem
    houve_mudanca = True
    while houve_mudanca:
        houve_mudanca = False
        for condicoes, conclusao in regras:
            if conclusao not in fatos and all(c in fatos for c in condicoes):
                fatos.add(conclusao)
                origem[conclusao] = condicoes
                houve_mudanca = True
    return fatos, origem


def por_que(objetivo, origem, nivel=0):
    prefixo = '  ' * nivel
    if objetivo not in origem:
        print(prefixo + '- ' + objetivo + ' (não derivado a partir dos fatos atuais)')
        return
    condicoes = origem[objetivo]
    if condicoes is None:
        print(prefixo + '- ' + objetivo + ' (fato inicial)')
    else:
        print(prefixo + '- ' + objetivo + ' porque:')
        for c in condicoes:
            por_que(c, origem, nivel + 1)


fatos, origem = encadeamento_com_justificativa(fatos_iniciais, regras)
print('Por que sugerir problemas avançados de álgebra?')
por_que('sugerir problemas avançados de álgebra', origem)

A saída mostra a árvore de justificativa em vários níveis: a sugestão de
problemas avançados se apoia em poder estudar álgebra, que por sua vez vem de dominar
aritmética, e em gostar de desafios. Para o projeto, troque os fatos iniciais, peça o
porquê de outras conclusões e, como desafio, ajuste os fatos para que o aluno chegue
a poder estudar cálculo.